In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from datetime import datetime
import tushare as ts
import scipy.stats as stats
import matplotlib.pyplot as plt

# 加入这两行配置
plt.rcParams['font.sans-serif'] = ['Arial Unicode MS', 'PingFang SC', 'SimHei'] 
plt.rcParams['axes.unicode_minus'] = False

# ==========================================
# 0. 核心配置 (动态波段参数)
# ==========================================
INIT_CAPITAL = 1000000.0   
CONTRACT_LIMIT = 1         
MULTIPLIER = 200           
MARGIN_RATE = 0.15         
IDLE_INTEREST = 0.011      
ENTRY_THRESHOLD = -0.035   # 入场：偏离度 < -3.5%
EXIT_THRESHOLD = -0.015    # 出场：偏离度 > -1.5%
FUT_CODE = 'IC2606.CFX'
SPOT_CODE = '000905.SH'
DELIVERY_DATE = '20260619'
FALLBACK_Q = 0.012  

pro = ts.pro_api()

# ==========================================
# 1. 数据准备 (T-1 滞后 + 容灾逻辑)
# ==========================================
def prepare_backtest_data_t1():
    print("正在拉取行情与利率数据...")
    df_fut = pro.fut_daily(ts_code=FUT_CODE, start_date='20250901', end_date='20260316')
    df_spot = pro.index_daily(ts_code=SPOT_CODE, start_date='20250901', end_date='20260316')
    df_shibor = pro.shibor(start_date='20250901', end_date='20260316').rename(columns={'date': 'trade_date', '1y': 'r_raw'})
    
    try:
        df_q = pro.index_dailybasic(ts_code=SPOT_CODE, start_date='20250901', end_date='20260316', fields='trade_date,dividend_yield')
        if 'dividend_yield' in df_q.columns and not df_q['dividend_yield'].isnull().all():
            df_q = df_q.rename(columns={'dividend_yield': 'q_raw'})
        elif 'dividend_yield_ttm' in df_q.columns:
            df_q = df_q.rename(columns={'dividend_yield_ttm': 'q_raw'})
        else:
            df_q = df_spot[['trade_date']].copy()
            df_q['q_raw'] = FALLBACK_Q * 100
    except:
        df_q = df_spot[['trade_date']].copy()
        df_q['q_raw'] = FALLBACK_Q * 100

    data = pd.merge(df_fut[['trade_date', 'close']], df_spot[['trade_date', 'close']], on='trade_date', suffixes=('_f', '_s'))
    data = pd.merge(data, df_shibor[['trade_date', 'r_raw']], on='trade_date', how='left')
    data = pd.merge(data, df_q[['trade_date', 'q_raw']], on='trade_date', how='left')
    data = data.sort_values('trade_date').reset_index(drop=True)

    # T-1 滞后：决策只能基于昨日数据
    data['r_lag'] = data['r_raw'].shift(1).ffill() / 100.0
    data['q_lag'] = data['q_raw'].shift(1).ffill() / 100.0
    
    delivery_dt = datetime.strptime(DELIVERY_DATE, '%Y%m%d')
    data['T_val'] = data['trade_date'].apply(lambda x: (delivery_dt - datetime.strptime(x, '%Y%m%d')).days / 365.0)

    # FV 与偏离度计算
    data['fv_lag'] = data['close_s'] * np.exp((data['r_lag'] - data['q_lag']) * data['T_val'])
    data['dev_lag'] = (data['close_f'] - data['fv_lag']) / data['fv_lag']
    
    return data.dropna().reset_index(drop=True)

# ==========================================
# 2. 回测引擎
# ==========================================
def run_backtest_engine(data, is_benchmark=False):
    cash = INIT_CAPITAL
    position = 0
    prev_fut_price = 0
    history = []
    
    for i, row in data.iterrows():
        today_fut = row['close_f']
        dev = row['dev_lag']
        
        # MTM
        if position == 1:
            cash += (today_fut - prev_fut_price) * MULTIPLIER
        
        # 闲置资金计息
        margin = today_fut * MULTIPLIER * MARGIN_RATE if position == 1 else 0
        cash += max(0, cash - margin) * (IDLE_INTEREST / 365.0)
        
        # 决策逻辑
        if not is_benchmark:
            if position == 0 and dev < ENTRY_THRESHOLD: position = 1
            elif position == 1 and dev > EXIT_THRESHOLD: position = 0
        else:
            position = 1 
            
        history.append({'trade_date': row['trade_date'], 'nav': cash, 'pos': position, 'dev': dev})
        prev_fut_price = today_fut
        
    return pd.DataFrame(history)

# ==========================================
# 3. 运行、评估与可视化
# ==========================================
try:
    processed_data = prepare_backtest_data_t1()
    strat_nav = run_backtest_engine(processed_data, is_benchmark=False)
    bench_nav = run_backtest_engine(processed_data, is_benchmark=True)

    # 收益与 Alpha 计算
    strat_total_return = (strat_nav['nav'].iloc[-1] / INIT_CAPITAL) - 1
    bench_total_return = (bench_nav['nav'].iloc[-1] / INIT_CAPITAL) - 1
    strat_ret = strat_nav['nav'].pct_change().fillna(0)
    index_ret = processed_data['close_s'].pct_change().fillna(0)
    
    beta, _, _, _, _ = stats.linregress(index_ret, strat_ret)
    alpha = strat_total_return - beta * (processed_data['close_s'].iloc[-1]/processed_data['close_s'].iloc[0]-1)

    print("\n" + "="*45)
    print(f"动态波段策略总盈利: {strat_total_return:.2%} (最终净值: {strat_nav['nav'].iloc[-1]:,.2f})")
    print(f"死扛基准总盈利: {bench_total_return:.2%} (最终净值: {bench_nav['nav'].iloc[-1]:,.2f})")
    print("-" * 45)
    print(f"Beta (杠杆敏感度): {beta:.3f}")
    print(f"期间 Alpha (超额): {alpha:.2%}")
    print(f"日度胜率 (Beat Index): {((strat_ret.values > index_ret.values).sum() / len(index_ret)):.2%}")
    print("="*45)

    # ==========================================
    # 绘图：双线对比与分离持仓/空仓状态
    # ==========================================
    fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(12, 8), gridspec_kw={'height_ratios': [3, 1]})
    
    trade_dates = pd.to_datetime(strat_nav['trade_date'])
    pos_array = strat_nav['pos'].values
    
    # --- 图 1：净值对比与背景高亮 ---
    ax1.plot(trade_dates, strat_nav['nav'], label=f'Timing Strategy ({strat_total_return:.2%})', color='#1f77b4', linewidth=2)
    ax1.plot(trade_dates, bench_nav['nav'], label=f'IC2606 Buy & Hold ({bench_total_return:.2%})', color='gray', linestyle='--', alpha=0.8)
    
    ymin1, ymax1 = ax1.get_ylim()
    ax1.fill_between(trade_dates, ymin1, ymax1, where=(pos_array == 1), color='mistyrose', alpha=0.4, label='持仓 (Risk On)')
    ax1.fill_between(trade_dates, ymin1, ymax1, where=(pos_array == 0), color='aliceblue', alpha=0.6, label='空仓 (Risk Off)')
    ax1.set_ylim(ymin1, ymax1)
    
    ax1.set_title('NAV Comparison: Dynamic Timing vs Buy & Hold', fontsize=12)
    ax1.set_ylabel('Net Asset Value (NAV)')
    ax1.legend(loc='upper left', ncol=2)
    ax1.grid(True, alpha=0.3)
    
    # --- 图 2：偏离度与触发线 ---
    ax2.plot(trade_dates, strat_nav['dev'], label='Deviation (T-1)', color='purple', alpha=0.6)
    ax2.axhline(ENTRY_THRESHOLD, color='green', linestyle=':', label=f'Entry ({ENTRY_THRESHOLD*100}%)')
    ax2.axhline(EXIT_THRESHOLD, color='red', linestyle=':', label=f'Exit ({EXIT_THRESHOLD*100}%)')
    
    ymin2, ymax2 = ax2.get_ylim()
    ax2.fill_between(trade_dates, ymin2, ymax2, where=(pos_array == 1), color='mistyrose', alpha=0.4)
    ax2.fill_between(trade_dates, ymin2, ymax2, where=(pos_array == 0), color='aliceblue', alpha=0.6)
    ax2.set_ylim(ymin2, ymax2)
    
    ax2.set_ylabel('Basis Deviation')
    ax2.legend(loc='upper left', ncol=3, fontsize='small')
    ax2.grid(True, alpha=0.3)
    
    plt.xticks(rotation=45)
    plt.tight_layout()
    plt.show()

    # ==========================================
    # 4. 导出每日 NAV 与状态判定文件
    # ==========================================
    # 提取所需列并合并
    final_export = pd.merge(processed_data, strat_nav[['trade_date', 'nav', 'pos', 'dev']], on='trade_date')
    final_export = pd.merge(final_export, bench_nav[['trade_date', 'nav']].rename(columns={'nav':'bench_nav'}), on='trade_date')
    
    # 添加明文状态标签
    final_export['持仓状态'] = final_export['pos'].apply(lambda x: '持仓' if x == 1 else '空仓')
    
    # 整理列顺序
    export_cols = [
        'trade_date', 'close_s', 'close_f', 'r_lag', 'q_lag', 
        'dev', '持仓状态', 'nav', 'bench_nav'
    ]
    final_export = final_export[export_cols].rename(columns={'nav': 'strat_nav', 'dev': 'decision_dev'})
    
    # 输出文件
    final_export.to_excel("ic2606_daily_nav.xlsx", index=False)
    print("每日 NAV 及明细已导出至: ic2606_daily_nav.xlsx")

except Exception as e:
    print(f"执行失败: {e}")